# FLAG 2027 — NB-2: ablation phases P0-P6 on the re-extracted features

Plan: `docs/PLAN_KAGGLE.md` §5. **One variable per run.** Every phase carries its winner forward only if
it beats the incumbent by >= 0.3 proxy (noise floor across splits); otherwise the incumbent stays.

**Settings:** GPU T4 (x1 is enough), **Internet OFF**. Inputs: `flag2027-feats-v2` (NB-1 output) **and**
`flag2027-features` (organiser CSVs, for the control run).

* Selection metric: `proxy = (int_ng + int_g) / 2` over 3 speaker-disjoint splits x 3 models; P4 uses `int_g`.
* `int_g_short`: gender-protocol EER when val voices are cropped to Bangla-like length (only for encoders with crops).
* Internal validation is English-only. **P5 (Bangla) is decided on CodaBench**, never internally.
* Results go to `ablation.csv` after every run; re-running the notebook resumes (finished runs are cached).

In [1]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

KAGGLE | input /kaggle/input | working /kaggle/working


In [2]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_v2.py', 'wb').write(base64.b64decode(''.join('''
...
Y2hlY2twb2ludCwgZG9jcy9EQVRBLm1kKSIpCg==
'''.split())))
print('written:', ['flag_lib.py', 'flag_v2.py'])

written: ['flag_lib.py', 'flag_v2.py']


In [3]:
import numpy as np, pandas as pd, json, torch
os.environ["FLAG_DATA"] = str(IN)
import flag_v2 as V
V.polarity_preflight()
store = V.FeatureStore(IN)
T = V.Table(WORKDIR / "ablation.csv")
# budget: 3 splits x 3 models per run on Kaggle; 1 x 1 with few epochs for the local smoke test
KW = dict(seeds=(1,), n_models=1) if LOCAL else dict(seeds=(1, 2, 3), n_models=3)
EP = dict(epochs=2) if LOCAL else {}
HAS = {n: store.has("voice", n) for n in ["ecapa192", "ecapa6144"]}
HAS["arcface"] = store.has("face", "arcface")
SSL = [p.name.split("_")[1] for p in (store.fd.glob("voice_*_train.npy") if store.fd else [])
       if p.name.split("_")[1] not in ("ecapa192", "ecapa6144", "ecapa192crop", "ecapa6144crop")]
print("available:", HAS, "| ssl:", SSL, "| device", V.DEVICE)

13:27:51 polarity: internal scores are higher = same; write_submission writes -score (LOWER = same, measured on CodaBench: -d2 57.83 vs +d2 42.17 with one checkpoint, docs/DATA.md)
13:29:59 feature dir: /kaggle/input/datasets/chnnhnnguyn/flag2027-feats-v2/feats | 33 arrays
available: {'ecapa192': True, 'ecapa6144': True, 'arcface': True} | ssl: ['wavlm', 'xlsr'] | device cuda


## P0 — feature sanity + unimodal quality

In [4]:
# P0 - are the new features sane, and how good is each one on its own (no training)?
rows = []
if HAS["ecapa192"]:
    a, b = store.voice("given", "train"), store.voice("ecapa192", "train")
    c = np.sum(V.l2n(a) * V.l2n(b), 1)
    print(f"cos(own ECAPA-192, organiser 192): mean {c.mean():.4f}  p5 {np.percentile(c, 5):.4f}")
for kind, name in [("face", "vgg"), ("face", "arcface"), ("voice", "given"), ("voice", "ecapa192"), ("voice", "ecapa6144")]:
    if kind == "face" and name == "arcface" and not HAS["arcface"]: continue
    if kind == "voice" and name != "given" and not HAS[name]: continue
    X_ = store.face(name, "train") if kind == "face" else store.voice(name, "train")
    rows.append(dict(kind=kind, feat=name, dim=X_.shape[1], **V.unimodal_eer(X_, store.spk, store.gmap)))
    C = store.voice_crops(name) if kind == "voice" else None
    if C is not None:
        rows.append(dict(kind=kind, feat=name + " (Bangla-length crop)", dim=C.shape[2], **V.unimodal_eer(C[0], store.spk, store.gmap)))
# SSL: pick the best block of layers by unimodal speaker EER (cheap, no training)
BEST_SSL = {}
for s in SSL:
    nl = np.load(store.fd / f"voice_{s}_train.npy", mmap_mode="r").shape[1]
    wins = sorted({(a, min(a + w - 1, nl - 1)) for w in (3, 6) for a in range(1, nl, 3)})
    best = None
    for a, b in wins:
        r = V.unimodal_eer(store.voice(f"{s}_L{a}-{b}", "train"), store.spk, store.gmap)
        rows.append(dict(kind="voice", feat=f"{s}_L{a}-{b}", dim=None, **r))
        if best is None or r["g"] < best[1]: best = (f"{s}_L{a}-{b}", r["g"])
    BEST_SSL[s] = best[0]
P0 = pd.DataFrame(rows); P0.to_csv(WORKDIR / "p0_unimodal.csv", index=False)
print(P0.sort_values(["kind", "g"]).to_string(index=False))
print("best SSL layer blocks:", BEST_SSL)

cos(own ECAPA-192, organiser 192): mean 0.0183  p5 -0.0999
 kind                           feat    dim    ng     g
 face                        arcface  512.0  1.70  1.87
 face                            vgg 4096.0  7.77  9.73
voice                       ecapa192  192.0  4.33  5.33
voice  ecapa192 (Bangla-length crop)  192.0  4.60  5.87
voice                          given  192.0  4.77  5.93
voice                      ecapa6144 6144.0  8.97 11.30
voice ecapa6144 (Bangla-length crop) 6144.0  9.87 12.10
voice                     wavlm_L4-9    NaN 11.17 12.57
voice                     wavlm_L4-6    NaN 11.63 12.60
voice                     wavlm_L7-9    NaN 11.60 12.80
voice                     wavlm_L1-6    NaN 12.03 13.27
voice                    wavlm_L7-12    NaN 13.43 14.53
voice                      xlsr_L4-6    NaN 13.60 14.53
voice                     wavlm_L1-3    NaN 13.43 15.37
voice                      xlsr_L1-6    NaN 13.93 15.60
voice                      xlsr_L4-9    NaN 1

## P1 — head / regularisation (organiser features)

In [5]:
# P1 - head/regularisation on the ORGANISER features (control = our current recipe)
CTRL = dict(dict(head="mlp", drop=0.3, emb=128, epochs=40), **EP)
T.run(store, "P1", "P1 control mlp drop.3 emb128", CTRL, **KW)
T.run(store, "P1", "P1 linear drop.3", dict(CTRL, head="linear"), **KW)
T.run(store, "P1", "P1 linear drop.9", dict(CTRL, head="linear", drop=0.9), **KW)
T.run(store, "P1", "P1 linear drop.9 emb192", dict(CTRL, head="linear", drop=0.9, emb=192), **KW)
T.run(store, "P1", "P1 mlp drop.9", dict(CTRL, drop=0.9), **KW)
CFG, INC = V.pick(T, "P1", "P1 control mlp drop.3 emb128")

13:34:28 [P1] P1 control mlp drop.3 emb128                  ng  21.83  g  30.49  proxy  26.16  (243s)
13:34:57 [P1] P1 linear drop.3                              ng  22.70  g  31.54  proxy  27.12  (29s)
13:35:26 [P1] P1 linear drop.9                              ng  24.19  g  33.91  proxy  29.05  (29s)
13:35:55 [P1] P1 linear drop.9 emb192                       ng  24.32  g  33.98  proxy  29.15  (28s)
13:36:31 [P1] P1 mlp drop.9                                 ng  25.24  g  37.41  proxy  31.33  (37s)
13:36:31 [P1] keep incumbent P1 control mlp drop.3 emb128 (proxy 26.16; best challenger P1 control mlp drop.3 emb128 26.16)


## P2 — features (the decisive test)

In [6]:
# P2 - which feature is the bottleneck?  Same head/loss as the P1 winner, only the features change.
feats = [("vgg", "given", "A vgg + given192 (control)")]
if HAS["ecapa6144"]: feats.append(("vgg", "ecapa6144", "B vgg + ecapa6144"))
if HAS["ecapa192"]:  feats.append(("vgg", "ecapa192", "C vgg + own ecapa192"))
if HAS["arcface"]:   feats.append(("arcface", "given", "D arcface + given192"))
if HAS["arcface"] and HAS["ecapa6144"]: feats.append(("arcface", "ecapa6144", "E arcface + ecapa6144"))
for s, blk in BEST_SSL.items():
    feats.append(("vgg", blk, f"F vgg + {blk}"))
    if HAS["arcface"]: feats.append(("arcface", blk, f"G arcface + {blk}"))
names = {}
for f, v, nm in feats:
    name = f"P2 {nm}"
    T.run(store, "P2", name, dict(CFG, face=f, voice=v), **KW); names[nm] = name
# control row equals the P1 incumbent config on organiser features
CFG, INC = V.pick(T, "P2", names["A vgg + given192 (control)"])
print(T.df[T.df.phase == "P2"][["run", "int_ng", "int_g", "proxy", "sd_g"]].to_string(index=False))

13:37:08 [P2] P2 A vgg + given192 (control)                 ng  21.83  g  30.49  proxy  26.16  (37s)
13:42:37 [P2] P2 B vgg + ecapa6144                          ng  20.56  g  29.26  proxy  24.91  short 30.53  (330s)
13:44:55 [P2] P2 C vgg + own ecapa192                       ng  22.10  g  30.26  proxy  26.18  short 32.5  (137s)
13:45:29 [P2] P2 D arcface + given192                       ng  29.80  g  38.04  proxy  33.92  (34s)
13:49:15 [P2] P2 E arcface + ecapa6144                      ng  26.33  g  35.80  proxy  31.07  short 36.34  (226s)
13:51:38 [P2] P2 F vgg + wavlm_L4-9                         ng  22.12  g  31.10  proxy  26.61  (143s)
13:52:15 [P2] P2 G arcface + wavlm_L4-9                     ng  26.83  g  35.63  proxy  31.23  (38s)
13:54:38 [P2] P2 F vgg + xlsr_L4-6                          ng  24.60  g  34.59  proxy  29.59  (143s)
13:55:15 [P2] P2 G arcface + xlsr_L4-6                      ng  28.94  g  37.18  proxy  33.06  (38s)
13:55:15 [P2] winner: P2 B vgg + ecapa6144 (prox

## P3 — loss

In [7]:
# P3 - loss: add shared-centre AAM, then explicit alignment, then orthogonality (one at a time)
base_name = INC
T.run(store, "P3", "P3 = incumbent", CFG, **KW)
for m, s in [(0.2, 16.0), (0.2, 32.0), (0.3, 32.0)]:
    T.run(store, "P3", f"P3 +AAM m{m} s{int(s)}", dict(CFG, w_aam=1.0, margin=m, scale=s), **KW)
CFG_A, INC_A = V.pick(T, "P3", "P3 = incumbent")
T.run(store, "P3", "P3 winner +mse1", dict(CFG_A, w_mse=1.0), **KW)
T.run(store, "P3", "P3 winner +orth.5", dict(CFG_A, w_orth=0.5), **KW)
T.run(store, "P3", "P3 AAM only (no InfoNCE)", dict(CFG_A, w_nce=0.0, w_aam=1.0), **KW)
CFG_STD, INC_STD = V.pick(T, "P3", INC_A)
print("STANDARD model:", INC_STD)

13:56:05 [P3] P3 = incumbent                                ng  20.56  g  29.26  proxy  24.91  short 30.53  (50s)
13:57:12 [P3] P3 +AAM m0.2 s16                              ng  20.41  g  29.04  proxy  24.73  short 30.3  (67s)
13:58:19 [P3] P3 +AAM m0.2 s32                              ng  20.33  g  28.97  proxy  24.65  short 29.96  (66s)
13:59:25 [P3] P3 +AAM m0.3 s32                              ng  20.24  g  28.89  proxy  24.57  short 29.86  (66s)
13:59:25 [P3] winner: P3 +AAM m0.3 s32 (proxy 24.57 vs incumbent 24.91)
14:00:33 [P3] P3 winner +mse1                               ng  20.59  g  29.32  proxy  24.96  short 30.54  (69s)
14:01:46 [P3] P3 winner +orth.5                             ng  20.27  g  29.03  proxy  24.65  short 29.96  (73s)
14:02:46 [P3] P3 AAM only (no InfoNCE)                      ng  21.38  g  29.57  proxy  25.47  short 30.73  (60s)
14:02:46 [P3] keep incumbent P3 +AAM m0.3 s32 (proxy 24.57; best challenger P3 +AAM m0.3 s32 24.57)
STANDARD model: P3 +AAM m0.3 s3

## P4 — Gender model

In [8]:
# P4 - Gender model for the gender/* cells: judged on int_g only
T.run(store, "P4", "P4 = standard", CFG_STD, **KW)
T.run(store, "P4", "P4 same-gender negatives", dict(CFG_STD, same_gender_only=True), **KW)
CFG_SG, INC_SG = V.pick(T, "P4", "P4 = standard", key="int_g")
for lam in (0.05, 0.2):
    T.run(store, "P4", f"P4 winner +GRL gender {lam}", dict(CFG_SG, grl_gender=lam), **KW)
CFG_GEN, INC_GEN = V.pick(T, "P4", INC_SG, key="int_g")
print("GENDER model:", INC_GEN)

14:03:53 [P4] P4 = standard                                 ng  20.24  g  28.89  proxy  24.57  short 29.86  (67s)
14:05:00 [P4] P4 same-gender negatives                      ng  22.32  g  29.21  proxy  25.77  short 30.67  (68s)
14:05:00 [P4] keep incumbent P4 = standard (int_g 28.89; best challenger P4 = standard 28.89)
14:06:13 [P4] P4 winner +GRL gender 0.05                    ng  22.76  g  29.27  proxy  26.01  short 30.77  (72s)
14:07:24 [P4] P4 winner +GRL gender 0.2                     ng  22.62  g  29.21  proxy  25.92  short 30.93  (71s)
14:07:24 [P4] keep incumbent P4 = standard (int_g 28.89; best challenger P4 = standard 28.89)
GENDER model: P4 = standard


## P5 — Bangla (decided on CodaBench)

In [9]:
# P5 - Bangla. Internal val cannot see Bangla, so these become CodaBench submissions.
# (int_g_short is reported as a weak hint: English val voices cropped to Bangla-like length.)
has_crops = store.voice_crops(CFG_STD["voice"]) is not None
P5 = {"base": (CFG_STD, CFG_GEN)}
if has_crops:
    for view in ("mix", "crops"):
        T.run(store, "P5", f"P5 std voice_view={view}", dict(CFG_STD, voice_view=view), **KW)
    P5["crops_mix"] = (dict(CFG_STD, voice_view="mix"), dict(CFG_GEN, voice_view="mix"))
T.run(store, "P5", "P5 std +DANN language .1 (uses dev audio)", dict(CFG_STD, dann_lang=0.1), **KW)
P5["dann"] = (dict(CFG_STD, dann_lang=0.1), dict(CFG_GEN, dann_lang=0.1))
print(T.df[T.df.phase == "P5"][["run", "int_ng", "int_g", "proxy"] + (["int_g_short"] if "int_g_short" in T.df else [])].to_string(index=False))

14:08:36 [P5] P5 std voice_view=mix                         ng  20.27  g  28.70  proxy  24.48  short 29.68  (71s)
14:09:46 [P5] P5 std voice_view=crops                       ng  20.30  g  28.96  proxy  24.63  short 30.01  (71s)
14:11:11 [P5] P5 std +DANN language .1 (uses dev audio)     ng  21.08  g  29.58  proxy  25.33  short 30.88  (85s)
                                      run  int_ng  int_g  proxy  int_g_short
                    P5 std voice_view=mix   20.27  28.70  24.48        29.68
                  P5 std voice_view=crops   20.30  28.96  24.63        30.01
P5 std +DANN language .1 (uses dev audio)   21.08  29.58  25.33        30.88


## P6 — fusion weight + candidate submissions

In [10]:
# P6 - CCA fusion weight (internal), then write the candidate submissions
for w in (0.0, 0.15, 0.35):
    T.run(store, "P6", f"P6 std cca_w={w}", CFG_STD, cca_w=w, **KW)
d6 = T.df[T.df.run.str.startswith("P6") | (T.df.run == INC_STD)]
W_BEST = 0.25
best6 = T.df[T.df.phase == "P6"].sort_values("proxy").iloc[0]
if best6.proxy <= T.get(INC_STD)["proxy"] - 0.3:
    W_BEST = float(best6.run.split("=")[1])
print("CCA weight:", W_BEST)

N_FINAL = 2 if LOCAL else 10
SUBS = WORKDIR / "submissions"; SUBS.mkdir(exist_ok=True)
MANI = []
for tag_, (cs, cg) in P5.items():
    std = V.dev_scores(store, cs, n_models=N_FINAL, cca_w=W_BEST)
    gen = std if cg == cs else V.dev_scores(store, cg, n_models=N_FINAL, cca_w=W_BEST)
    p = SUBS / f"sub_{tag_}.zip"
    V.write_zip(store, p, V.merge_cells(std, gen)); V.check_zip(store, p)
    MANI.append(dict(zip=p.name, standard=json.dumps(cs), gender=json.dumps(cg), cca_w=W_BEST))
pd.DataFrame(MANI).to_csv(SUBS / "submissions.csv", index=False)
print(pd.DataFrame(MANI)[["zip", "cca_w"]])

14:12:18 [P6] P6 std cca_w=0.0                              ng  22.04  g  29.39  proxy  25.72  short 29.86  (67s)
14:13:25 [P6] P6 std cca_w=0.15                             ng  20.54  g  28.84  proxy  24.69  short 29.86  (67s)
14:14:32 [P6] P6 std cca_w=0.35                             ng  20.13  g  29.23  proxy  24.68  short 29.86  (67s)
CCA weight: 0.25
14:25:30 zip OK: /kaggle/working/submissions/sub_base.zip
14:35:56 zip OK: /kaggle/working/submissions/sub_crops_mix.zip
14:46:38 zip OK: /kaggle/working/submissions/sub_dann.zip
                 zip  cca_w
0       sub_base.zip   0.25
1  sub_crops_mix.zip   0.25
2       sub_dann.zip   0.25


## Summary

In [11]:
print(T.df[["phase", "run", "int_ng", "int_g", "proxy", "sd_g", "sec"]].to_string(index=False))
print("""
Submit to CodaBench, in this order (one variable each):
  1. submissions/sub_base.zip       - new features + P1-P4 winners      (vs 30.90)
  2. submissions/sub_crops_mix.zip  - + duration-matched voice crops    (read the two Bangla cells)
  3. submissions/sub_dann.zip       - + DANN language on dev audio      (transductive: note it in the report)
Then per-cell merge the best cells with NB-3 (FLAG_06_submit).""")

phase                                       run  int_ng  int_g  proxy  sd_g  sec
   P1              P1 control mlp drop.3 emb128   21.83  30.49  26.16  2.21  243
   P1                          P1 linear drop.3   22.70  31.54  27.12  3.83   29
   P1                          P1 linear drop.9   24.19  33.91  29.05  2.36   29
   P1                   P1 linear drop.9 emb192   24.32  33.98  29.15  2.45   28
   P1                             P1 mlp drop.9   25.24  37.41  31.33  3.07   37
   P2             P2 A vgg + given192 (control)   21.83  30.49  26.16  2.21   37
   P2                      P2 B vgg + ecapa6144   20.56  29.26  24.91  1.28  330
   P2                   P2 C vgg + own ecapa192   22.10  30.26  26.18  1.05  137
   P2                   P2 D arcface + given192   29.80  38.04  33.92  3.94   34
   P2                  P2 E arcface + ecapa6144   26.33  35.80  31.07  0.86  226
   P2                     P2 F vgg + wavlm_L4-9   22.12  31.10  26.61  1.68  143
   P2                 P2 G a